# 🗂️ Module 3.2: Create & Manage Experiments

**Goal**: Learn to create, organize, tag, and manage experiments programmatically.

---

In [ ]:
import mlflow
from mlflow.entities import ViewType
import pandas as pd

mlflow.autolog(disable=True)
print(f"MLFlow version: {mlflow.__version__}")
print(f"Tracking URI: {mlflow.get_tracking_uri()}")

## 1. Creating Experiments

Two ways to create experiments: explicit `create_experiment()` or implicit `set_experiment()`.

In [ ]:
# Method 1: Explicit creation with tags
exp_id = mlflow.create_experiment(
    name="03_experiment_demo",
    tags={
        "team": "ml-engineering",
        "project": "wine-classification",
        "version": "1.0"
    }
)
print(f"✅ Created experiment with ID: {exp_id}")

In [ ]:
# Method 2: set_experiment — creates if doesn't exist, selects if it does
mlflow.set_experiment("03_experiment_auto_created")
print("✅ Experiment set (created automatically if it didn't exist)")

# This is the simpler approach and what most people use

## 2. Getting Experiment Info

In [ ]:
# Get experiment by name
exp = mlflow.get_experiment_by_name("03_experiment_demo")

print("📋 Experiment Details:")
print(f"   Name: {exp.name}")
print(f"   ID: {exp.experiment_id}")
print(f"   Artifact Location: {exp.artifact_location}")
print(f"   Lifecycle Stage: {exp.lifecycle_stage}")
print(f"   Tags: {exp.tags}")

In [ ]:
# Get experiment by ID
exp_by_id = mlflow.get_experiment(exp_id)
print(f"Got experiment: {exp_by_id.name} (by ID={exp_id})")

## 3. Listing All Experiments

In [ ]:
# List all active experiments
experiments = mlflow.search_experiments()

print("📋 All Active Experiments:")
print("=" * 70)
for exp in experiments:
    print(f"  ID={exp.experiment_id:>3} | {exp.name}")

print(f"\nTotal: {len(experiments)} experiments")

In [ ]:
# Search experiments with filters
filtered = mlflow.search_experiments(
    filter_string="tags.team = 'ml-engineering'"
)

print(f"Experiments tagged 'ml-engineering': {len(filtered)}")
for exp in filtered:
    print(f"  - {exp.name}")

## 4. Adding Runs to an Experiment

In [ ]:
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

wine = load_wine()
X_train, X_test, y_train, y_test = train_test_split(
    wine.data, wine.target, test_size=0.2, random_state=42
)

# Use our created experiment
mlflow.set_experiment("03_experiment_demo")

# Add some runs
for n_est in [50, 100, 200]:
    with mlflow.start_run(run_name=f"rf_{n_est}_trees"):
        mlflow.log_param("n_estimators", n_est)
        model = RandomForestClassifier(n_estimators=n_est, random_state=42)
        model.fit(X_train, y_train)
        acc = accuracy_score(y_test, model.predict(X_test))
        mlflow.log_metric("accuracy", acc)
        print(f"  n_estimators={n_est}: accuracy={acc:.4f}")

print("\n✅ Runs added to '03_experiment_demo'")

## 5. Experiment Lifecycle (Delete & Restore)

In [ ]:
# Create a temporary experiment
temp_id = mlflow.create_experiment("03_temp_experiment")
print(f"Created temp experiment: ID={temp_id}")

# Soft delete (moves to 'deleted' state — can be restored!)
mlflow.delete_experiment(temp_id)
print("Deleted (soft delete)")

# View deleted experiments
deleted = mlflow.search_experiments(view_type=ViewType.DELETED_ONLY)
print(f"Deleted experiments: {[e.name for e in deleted]}")

# Restore it
# Note: The restore API is available via the MlflowClient
from mlflow import MlflowClient
client = MlflowClient()
client.restore_experiment(temp_id)
print("Restored!")

# Verify it's back
exp = mlflow.get_experiment(temp_id)
print(f"Lifecycle stage: {exp.lifecycle_stage}")

# Clean up — delete again
mlflow.delete_experiment(temp_id)
print("\n✅ Experiment lifecycle demo complete!")

## 🔑 Key Takeaways

1. **`set_experiment()`** is the simplest way — it creates or selects
2. **`create_experiment()`** gives you more control (tags, artifact location)
3. **Experiment tags** help organize experiments by team, project, or phase
4. **`search_experiments()`** finds experiments by tags or name
5. **Deleting is a soft delete** — experiments can be restored

---

## ➡️ Next: `03_search_filter_runs.ipynb` — Power searching & filtering